# Notebook 1: Earnings Call Transcript Scraping

Scrapes earnings call transcripts from The Motley Fool for 6 tickers across 4 quarters (2024).

**Output:** Raw HTML saved to `data/raw/transcripts/`, cleaned text saved to `data/processed/transcripts_clean.csv`

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import re
import json
from datetime import datetime

In [16]:
# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DIR       = 'data/raw/transcripts'
PROCESSED_DIR = 'data/processed'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Directories ready.')

Directories ready.


## 1. Define Tickers & Known Earnings Dates

We hardcode the earnings announcement dates as a reliable anchor.

In [5]:
# ── Ticker metadata ─────────────────────────────────────────────────────────
TICKERS = {
    'AAPL': {'name': 'apple',          'sector': 'Technology'},
    'MSFT': {'name': 'microsoft',      'sector': 'Technology'},
    'NVDA': {'name': 'nvidia',         'sector': 'Technology'},
    'JPM':  {'name': 'jpmorgan-chase', 'sector': 'Finance'}
}

# ── Actual earnings announcement dates ──────────────────────────────────────
# Tuple format: (announcement_date, quarter, fiscal_year)
EARNINGS_DATES = {
    'AAPL': [
        ('2022-01-28', 'Q1', '2022'), # year 2022
        ('2022-04-29', 'Q2', '2022'),
        ('2022-07-28', 'Q3', '2022'),
        ('2022-10-27', 'Q4', '2022'),
    ],
    'MSFT': [
        ('2023-10-24', 'Q1', '2024'),  # year 2024
        ('2024-01-31', 'Q2', '2024'),
        ('2024-04-25', 'Q3', '2024'),
        ('2024-07-30', 'Q4', '2024'),
    ],
    'NVDA': [
        ('2024-05-29', 'Q1', '2025'),  # year 2025
        ('2024-08-28', 'Q2', '2025'),
        ('2024-11-20', 'Q3', '2025'),
        ('2025-02-26', 'Q4', '2025'),
    ],
    'JPM': [
        ('2024-04-12', 'Q1', '2024'),  # year 2024
        ('2024-07-12', 'Q2', '2024'),
        ('2024-10-11', 'Q3', '2024'),
        ('2025-01-15', 'Q4', '2024'),
    ],
}

print(f'Tickers: {list(TICKERS.keys())}')
print(f'Total earnings events: {sum(len(v) for v in EARNINGS_DATES.values())}')

Tickers: ['AAPL', 'MSFT', 'NVDA', 'JPM']
Total earnings events: 16


## 2. Build Transcript URLs

Motley Fool URL pattern (verified):
```
https://www.fool.com/earnings/call-transcripts/YYYY/MM/DD/companyname-slug-qN-YYYY-earnings-call-transcript/

```
We generate the primary URL + 1-2 fallback date variants (in case the transcript posts a day after earnings).

In [6]:
from datetime import timedelta

BASE_URL = 'https://www.fool.com/earnings/call-transcripts'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}


def build_url_candidates(ticker: str, date_str: str, quarter: str, fiscal_year: str = None) -> list[str]:
    name         = TICKERS[ticker]['name']
    ticker_lower = ticker.lower()
    path_year    = date_str[:4]
    slug_year    = fiscal_year if fiscal_year else path_year  # use fiscal year for slug if provided
    q_num        = quarter[1]
    base_date    = datetime.strptime(date_str, '%Y-%m-%d')

    candidates = []
    for delta in [0, 1, 2]:
        d     = base_date + timedelta(days=delta)
        slug  = f'{name}-{ticker_lower}-q{q_num}-{slug_year}-earnings-call-transcript'
        slug_truncated = slug[:-1]
        base  = f'{BASE_URL}/{d.year}/{d.month:02d}/{d.day:02d}'
        candidates.append(f'{base}/{slug}/')
        candidates.append(f'{base}/{slug_truncated}/')

    return candidates


# Preview URLs
sample = build_url_candidates('AAPL', '2022-01-28', 'Q1')
for u in sample:
    print(u)

https://www.fool.com/earnings/call-transcripts/2022/01/28/apple-aapl-q1-2022-earnings-call-transcript/
https://www.fool.com/earnings/call-transcripts/2022/01/28/apple-aapl-q1-2022-earnings-call-transcrip/
https://www.fool.com/earnings/call-transcripts/2022/01/29/apple-aapl-q1-2022-earnings-call-transcript/
https://www.fool.com/earnings/call-transcripts/2022/01/29/apple-aapl-q1-2022-earnings-call-transcrip/
https://www.fool.com/earnings/call-transcripts/2022/01/30/apple-aapl-q1-2022-earnings-call-transcript/
https://www.fool.com/earnings/call-transcripts/2022/01/30/apple-aapl-q1-2022-earnings-call-transcrip/


## 3. Scraper Function

In [7]:
def fetch_transcript(ticker: str, date_str: str, quarter: str,
                     fiscal_year: str = None,
                     sleep_sec: float = 2.0) -> dict | None:
    """
    Fetch and parse a Motley Fool earnings call transcript.
    Args:
        ticker:      Stock ticker symbol (e.g. 'AAPL')
        date_str:    Earnings date in 'YYYY-MM-DD' format
        quarter:     Quarter label e.g. 'Q1'
        fiscal_year: Fiscal year for slug (e.g. '2024'). Defaults to date year.
        sleep_sec:   Seconds to wait between requests (be polite!)
    Returns:
        dict with keys: ticker, date, quarter, fiscal_year, sector, url, full_text,
                        prepared_remarks, qa_section
        None if all candidate URLs fail.
    """
    candidates = build_url_candidates(ticker, date_str, quarter, fiscal_year=fiscal_year)

    for url in candidates:
        try:
            time.sleep(sleep_sec)
            resp = requests.get(url, headers=HEADERS, timeout=15)

            if resp.status_code == 404:
                continue
            if resp.status_code != 200:
                print(f'  [{ticker} {quarter}] HTTP {resp.status_code} — {url}')
                continue

            soup = BeautifulSoup(resp.text, 'html.parser')

            # ── Save raw HTML ────────────────────────────────────────────────
            fy = fiscal_year if fiscal_year else date_str[:4]
            fname = f'{ticker}_{quarter}_{fy}.html'
            with open(os.path.join(RAW_DIR, fname), 'w', encoding='utf-8') as f:
                f.write(resp.text)

            # ── Extract main article body ────────────────────────────────────
            body = soup.find('div', class_='article-body')
            if not body:
                body = soup.find('article') or soup

            paragraphs = body.find_all('p')
            full_text  = ' '.join(p.get_text(separator=' ', strip=True)
                                  for p in paragraphs)

            # ── Split into Prepared Remarks vs Q&A ──────────────────────────
            prepared, qa = split_transcript_sections(full_text)

            print(f'  ✓ {ticker} {quarter} FY{fy} — {len(full_text):,} chars | URL: {url}')

            return {
                'ticker':           ticker,
                'date':             date_str,
                'quarter':          quarter,
                'fiscal_year':      fy,
                'sector':           TICKERS[ticker]['sector'],
                'url':              url,
                'full_text':        full_text,
                'prepared_remarks': prepared,
                'qa_section':       qa,
                'char_count':       len(full_text),
            }

        except requests.exceptions.RequestException as e:
            print(f'  [ERROR] {ticker} {quarter} — {e}')
            continue

    print(f'  ✗ {ticker} {quarter} — all candidate URLs failed')
    return None


def split_transcript_sections(text: str) -> tuple[str, str]:
    """
    Split transcript into 'Prepared Remarks' and 'Questions & Answers' sections.
    Returns (prepared_text, qa_text). If no split found, returns (full_text, '').
    """
    # Common dividers used in Motley Fool transcripts
    dividers = [
        'Questions and Answers',
        'Question-and-Answer Session',
        'Q&A Session',
        'QUESTIONS AND ANSWERS',
    ]
    for divider in dividers:
        if divider.lower() in text.lower():
            idx = text.lower().index(divider.lower())
            return text[:idx].strip(), text[idx:].strip()
    
    return text, ''


print('Scraper functions defined.')

Scraper functions defined.


## 4. Run Scraper for All 24 Transcripts

This will take ~2–3 minutes (2 second sleep between requests). 
Progress is printed for each transcript.

In [8]:
results = []
failed  = []

for ticker, dates in EARNINGS_DATES.items():
    for date_str, quarter, fiscal_year in dates:
        record = fetch_transcript(ticker, date_str, quarter, fiscal_year=fiscal_year, sleep_sec=2.0)
        if record:
            results.append(record)
        else:
            failed.append({'ticker': ticker, 'quarter': quarter, 'date': date_str})

print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Scraped:  {len(results)} / 16')
print(f'Failed:   {len(failed)}')
if failed:
    print('Failed entries:', failed)

  ✓ AAPL Q1 FY2022 — 49,053 chars | URL: https://www.fool.com/earnings/call-transcripts/2022/01/28/apple-aapl-q1-2022-earnings-call-transcript/
  ✓ AAPL Q2 FY2022 — 50,212 chars | URL: https://www.fool.com/earnings/call-transcripts/2022/04/29/apple-aapl-q2-2022-earnings-call-transcript/
  ✓ AAPL Q3 FY2022 — 48,381 chars | URL: https://www.fool.com/earnings/call-transcripts/2022/07/28/apple-aapl-q3-2022-earnings-call-transcript/
  ✓ AAPL Q4 FY2022 — 49,061 chars | URL: https://www.fool.com/earnings/call-transcripts/2022/10/27/apple-aapl-q4-2022-earnings-call-transcript/
  ✓ MSFT Q1 FY2024 — 55,743 chars | URL: https://www.fool.com/earnings/call-transcripts/2023/10/24/microsoft-msft-q1-2024-earnings-call-transcript/
  ✓ MSFT Q2 FY2024 — 57,543 chars | URL: https://www.fool.com/earnings/call-transcripts/2024/01/31/microsoft-msft-q2-2024-earnings-call-transcript/
  ✓ MSFT Q3 FY2024 — 57,007 chars | URL: https://www.fool.com/earnings/call-transcripts/2024/04/25/microsoft-msft-q3-2024-earnin

In [ ]:
import pandas as pd

# Getting raw transcripts
# Load processed transcripts — URLs are already stored there
df = pd.read_csv('data/processed/transcripts_clean.csv')

for _, row in df.iterrows():
    ticker  = row['ticker']
    quarter = row['quarter']
    url     = row['url']
    year    = str(row['date'])[:4]  # extract year from date column
    fname   = f'{ticker}_{quarter}_{year}.html'
    fpath   = os.path.join(RAW_DIR, fname)

    try:
        time.sleep(2)
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code == 200:
            with open(fpath, 'w', encoding='utf-8') as f:
                f.write(resp.text)
            print(f'✓ Saved {fname}')
        else:
            print(f'✗ {ticker} {quarter} — HTTP {resp.status_code}')
    except Exception as e:
        print(f'✗ {ticker} {quarter} — {e}')

✓ Saved AAPL_Q1_2022.html


## 5. Clean Transcript Text

Remove boilerplate disclaimers, speaker labels, and special characters.

In [12]:
# ── Boilerplate phrases common in Fool transcripts ───────────────────────────
BOILERPLATE_PATTERNS = [
    r'This is a.*?transcript.*?\.',
    r'All participants.*?mode\.',
    r'Please note.*?recorded\.',
    r'Forward.looking statements.*?\.\s',
    r'Safe harbor.*?\.',
    r'Thank you for standing by\.',
    r'Good (morning|afternoon|evening), (ladies and gentlemen|everyone)\.',
    r'\[Operator Instructions\]',
    r'\[.*?\]',          # remove all [bracketed text] e.g. [laughter]
    r'Image source:.*?\.',
]


def clean_text(text: str) -> str:
    """
    Clean raw transcript text:
    - Remove boilerplate phrases
    - Remove speaker labels (e.g. 'Tim Cook -- CEO')
    - Normalize whitespace
    - Remove non-ASCII characters
    """
    # Remove boilerplate
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE | re.DOTALL)

    # Remove speaker labels: "First Last -- Title" at start of paragraph
    text = re.sub(r'^[A-Z][a-z]+ [A-Z][a-z]+(?: [A-Z][a-z]+)? -- [^\n]+', '', text,
                  flags=re.MULTILINE)

    # Remove non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply cleaning
for r in results:
    r['full_text_clean']        = clean_text(r['full_text'])
    r['prepared_remarks_clean'] = clean_text(r['prepared_remarks'])
    r['qa_section_clean']       = clean_text(r['qa_section'])

print('Cleaning complete.')

# Spot-check
if results:
    sample = results[0]
    print(f"\n{sample['ticker']} {sample['quarter']} preview (first 400 chars):")
    print(sample['full_text_clean'][:400])

Cleaning complete.

AAPL Q1 preview (first 400 chars):
Apple ( AAPL 2.15% ) Q12022 Earnings Call Jan 27, 2022 , 5:00 p.m. ET Operator Good day, and welcome to the Apple Q1 FY 2022 earnings conference call. Today's call is being recorded. At this time, for opening remarks and introductions, I would like to turn the call over to Tejas Gala, director of investor relations and corporate finance. Please go ahead. Tejas Gala -- Director, Investor Relations,


## 6. Save Processed Transcripts

In [13]:
# Build DataFrame with key columns
cols = ['ticker', 'date', 'quarter', 'sector', 'url',
        'char_count', 'full_text_clean', 'prepared_remarks_clean', 'qa_section_clean']

df_transcripts = pd.DataFrame(results)[cols]
df_transcripts['date'] = pd.to_datetime(df_transcripts['date'])

out_path = os.path.join(PROCESSED_DIR, 'transcripts_clean.csv')
df_transcripts.to_csv(out_path, index=False)

print(f'Saved {len(df_transcripts)} transcripts → {out_path}')
print(f'\nShape: {df_transcripts.shape}')
df_transcripts[['ticker', 'quarter', 'date', 'sector', 'char_count']].head(12)

Saved 16 transcripts → data/processed/transcripts_clean.csv

Shape: (16, 9)


,ticker,quarter,date,sector,char_count
0,AAPL,Q1,2022-01-28,Technology,49053
1,AAPL,Q2,2022-04-29,Technology,50212
2,AAPL,Q3,2022-07-28,Technology,48381
3,AAPL,Q4,2022-10-27,Technology,49061
4,MSFT,Q1,2023-10-24,Technology,55743
5,MSFT,Q2,2024-01-31,Technology,57543
6,MSFT,Q3,2024-04-25,Technology,57007
7,MSFT,Q4,2024-07-30,Technology,54855
8,NVDA,Q1,2024-05-29,Technology,51459
9,NVDA,Q2,2024-08-28,Technology,53766
